In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q rectools[torch] lightning-fabric

In [ ]:
import os
import glob
import json

import numpy as np
import pandas as pd

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import BERT4RecModel, PopularModel
from rectools.metrics import NDCG, HitRate
from lightning_fabric import seed_everything

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
SEED_VALUE = 42
seed_everything(SEED_VALUE, workers=True)

## 1. Load listening events

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path('/kaggle/input/datasets/cruorvult/botify-dataset')
RAW_DATA_DIR = PROJECT_ROOT / 'tmp'

listening_paths = sorted(glob.glob(str(RAW_DATA_DIR / 'data.json*')))
print(f'Found {len(listening_paths)} listening files in {RAW_DATA_DIR}')

listen_log = pd.concat(
    [pd.read_json(path, lines=True)[['user', 'track', 'timestamp', 'time']] for path in listening_paths],
    ignore_index=True,
)

listen_log = listen_log.sort_values(['timestamp', 'user', 'track']).reset_index(drop=True)
print(f'Raw events: {listen_log.shape[0]:,}')
listen_log.head()

## Listening time distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(listen_log['time'], bins=50, kde=True)
plt.title('Distribution of Listening Time (listen_log[time])')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

We see high peaks at the distribution edges 

In [ ]:
# Hybrid positive feedback rule.
# Strong listens are kept deterministically, while weaker listens are sampled
# according to a smooth probability derived from relative listening duration.
ALWAYS_KEEP_TOP_TAIL = listen_log['time'].median() + listen_log['time'].std()/3
LISTENING_PROBABILITY_TEMPERATURE = 2
MIN_SAMPLING_PROBABILITY = 1e-3

sampling_rng = np.random.default_rng(SEED_VALUE)

strong_listen_cutoff = listen_log['time'].quantile(ALWAYS_KEEP_TOP_TAIL)
strong_listen_mask = listen_log['time'] >= strong_listen_cutoff

# Convert listening completion into a sampling probability.
# The power transform is intentionally strict: 0.5 -> 0.125, 0.8 -> 0.512,
# 0.95 -> 0.857. This keeps high-intent listens much more often than noisy ones.
listen_sample_probability = (
    listen_log['time']
    .clip(lower=MIN_SAMPLING_PROBABILITY, upper=1.0)
    ** LISTENING_PROBABILITY_TEMPERATURE
)

weak_sample_mask = sampling_rng.random(len(listen_log)) < listen_sample_probability
hybrid_positive_mask = strong_listen_mask | weak_sample_mask

positive_events = listen_log.loc[
    hybrid_positive_mask,
    ['user', 'track', 'timestamp', 'time'],
].copy()

# Deduplicate repeated selected listens while keeping the latest timestamp and
# the strongest observed completion as the implicit interaction weight.
positive_events = (
    positive_events
    .sort_values(['user', 'track', 'timestamp'])
    .groupby(['user', 'track'], as_index=False)
    .agg(timestamp=('timestamp', 'max'), time=('time', 'max'))
    .sort_values('timestamp')
    .reset_index(drop=True)
)

positive_interactions = positive_events.rename(columns={
    'user': Columns.User,
    'track': Columns.Item,
    'timestamp': Columns.Datetime,
    'time': Columns.Weight,
})

kept_share = positive_interactions.shape[0] / listen_log.shape[0]
print(f'Always-keep completion quantile: {ALWAYS_KEEP_TOP_TAIL:.2f}')
print(f'Always-keep completion cutoff: {strong_listen_cutoff:.4f}')
print(f'Sampling temperature: {LISTENING_PROBABILITY_TEMPERATURE:.1f}')
print(f'Raw events marked strong: {strong_listen_mask.sum():,}')
print(f'Raw events sampled by probability: {weak_sample_mask.sum():,}')
print(f'Positive interactions after deduplication: {positive_interactions.shape[0]:,} / {listen_log.shape[0]:,} ({kept_share:.2%})')
positive_interactions.head()


## 3. Time-based validation split

In [ ]:
TRAIN_FRACTION = 0.80
cut_position = int(TRAIN_FRACTION * len(positive_interactions))

model_train_events = positive_interactions.iloc[:cut_position].copy()
model_valid_events = positive_interactions.iloc[cut_position:].copy()

known_users = set(model_train_events[Columns.User])
known_items = set(model_train_events[Columns.Item])
model_valid_events = model_valid_events[
    model_valid_events[Columns.User].isin(known_users)
    & model_valid_events[Columns.Item].isin(known_items)
].copy()

print('train:', model_train_events.shape, 'valid:', model_valid_events.shape)

## 4. Build item features

In [ ]:
track_catalog = (
    pd.read_json(RAW_DATA_DIR / 'tracks.json', lines=True)
    .drop_duplicates(subset=['track'])
    .rename(columns={'track': Columns.Item})
)

active_tracks = track_catalog.loc[
    track_catalog[Columns.Item].isin(model_train_events[Columns.Item])
].copy()

def as_feature_frame(source_df, column_name, feature_name, explode=False):
    feature_df = source_df[[Columns.Item, column_name]].copy()
    if explode:
        feature_df = feature_df.explode(column_name)
    feature_df = feature_df.rename(columns={Columns.Item: 'id', column_name: 'value'})
    feature_df['feature'] = feature_name
    return feature_df.dropna(subset=['value'])

track_side_features = pd.concat(
    [
        as_feature_frame(active_tracks, 'artist_id', 'artist'),
        as_feature_frame(active_tracks, 'genres', 'genre', explode=True),
        as_feature_frame(active_tracks, 'mood', 'mood'),
        as_feature_frame(active_tracks, 'artist_country', 'country'),
        as_feature_frame(active_tracks, 'year', 'release_year'),
    ],
    ignore_index=True,
)

print(f'Catalog tracks: {track_catalog.shape[0]:,}; tracks used for training: {active_tracks.shape[0]:,}')
track_side_features.head()

In [ ]:
train_rectools_data = Dataset.construct(
    interactions_df=model_train_events,
    item_features_df=track_side_features,
    cat_item_features=['artist', 'genre', 'mood', 'country', 'release_year'],
)

valid_rectools_data = Dataset.construct(
    interactions_df=model_valid_events,
    item_features_df=track_side_features,
    cat_item_features=['artist', 'genre', 'mood', 'country', 'release_year'],
)

## 5. Train baseline and BERT4Rec

In [ ]:
top_popular_model = PopularModel()
top_popular_model.fit(train_rectools_data)

validation_users = model_valid_events[Columns.User].unique()
popular_validation_recs = top_popular_model.recommend(
    users=validation_users,
    dataset=train_rectools_data,
    k=10,
    filter_viewed=True,
)

In [ ]:
sequence_encoder = BERT4RecModel(
    session_max_len=150,      
    mask_prob=0.2,            
    loss="softmax",
    n_factors=96,             
    n_blocks=2,               
    n_heads=4,
    dropout_rate=0.15,        
    lr=1e-3,                  
    batch_size=128,
    epochs=120,               
    verbose=1,
    deterministic=True,
)

In [ ]:
%%time
sequence_encoder.fit(train_rectools_data)

In [ ]:
bert_validation_recs = sequence_encoder.recommend(
    users=validation_users,
    dataset=train_rectools_data,
    k=10,
    filter_viewed=True,
    on_unsupported_targets='warn',
)

## 6. Generate item-to-item recommendations

In [ ]:
candidate_anchor_items = (
    model_train_events.groupby(Columns.Item)
    .size()
    .sort_values(ascending=False)
    .index
    .to_list()
)

raw_i2i_neighbors = sequence_encoder.recommend_to_items(
    target_items=candidate_anchor_items,
    dataset=train_rectools_data,
    k=100,
    filter_itself=True,
    items_to_recommend=None,
    on_unsupported_targets='warn',
)
raw_i2i_neighbors.head()

## 7. Diversify by excluding same-artist neighbours

In [ ]:
artist_lookup = active_tracks[[Columns.Item, 'artist_id']].drop_duplicates()

diverse_i2i_neighbors = raw_i2i_neighbors.merge(
    artist_lookup.rename(columns={Columns.Item: 'target_item_id', 'artist_id': 'anchor_artist_id'}),
    on='target_item_id',
    how='inner',
)

diverse_i2i_neighbors = diverse_i2i_neighbors.merge(
    artist_lookup.rename(columns={'artist_id': 'neighbor_artist_id'}),
    on=Columns.Item,
    how='inner',
)

diverse_i2i_neighbors = (
    diverse_i2i_neighbors
    .loc[lambda frame: frame['anchor_artist_id'] != frame['neighbor_artist_id']]
    .drop(columns=['anchor_artist_id', 'neighbor_artist_id'])
    .sort_values(['target_item_id', 'rank'])
    .groupby('target_item_id', as_index=False)
    .head(10)
)

diverse_i2i_neighbors['rank'] = diverse_i2i_neighbors.groupby('target_item_id').cumcount() + 1
print('raw rows:', raw_i2i_neighbors.shape[0], 'diverse rows:', diverse_i2i_neighbors.shape[0])
diverse_i2i_neighbors.head()

## 8. Export JSONL

In [ ]:
SAVING_FOLDER = Path('/kaggle/working')
def save_i2i_jsonl(recommendation_frame, output_path):
    output_path = Path(output_path)
    with output_path.open('w') as output_file:
        for anchor_id, group in recommendation_frame.groupby('target_item_id'):
            ordered_recs = group.sort_values('rank')[Columns.Item].tolist()
            output_file.write(json.dumps({'item_id': anchor_id, 'recommendations': ordered_recs}) + '\n')

EXPORT_PATH = SAVING_FOLDER / 'bert4rec.jsonl'
save_i2i_jsonl(diverse_i2i_neighbors, EXPORT_PATH)
print(EXPORT_PATH)